In [17]:
import json
import pandas as pd

In [18]:
path = 'task1_train.jsonl'
with open(path, 'r') as file:
    train_data = [json.loads(line) for line in file] 

In [19]:
train_df = pd.DataFrame(train_data)

In [20]:
train_df

,doc_id,fact,explanation,statute
0,2013.INSC.228.txt,The facts which are essential to be stated fo...,{'The facts which are essential to be stated f...,[IPC 147]
1,1996.INSC.902.txt,16 Praveen Chandra was the Divisional Forest ...,"{'When A-1 started the engine, A-4, A-6 and A-...",[IPC 147]
2,2003.INSC.675.txt,Law was set in motion by PW-1(Chanakya) on th...,{'Law was set in motion by PW-1(Chanakya) on t...,[IPC 147]
3,2009.INSC.116.txt,The incident leading to the prosecution of th...,{'It was the prosecution case that Mahabir son...,[IPC 147]
4,2002.INSC.528.txt,", the complainant Abdul Jabbar s/o Sheikh Mun...",{'As soon as he reached near the shop he heard...,[IPC 147]
...,...,...,...,...
520,2011.INSC.13.txt,The prosecution case is that the victim Nand...,{'On 1994 at about 00 P.M. when the victim Nan...,[IPC 506]
521,2013.INSC.597.txt,The complainant/respondent no.2 herein (Deep...,{'The complainant/respondent no.2 herein (Deep...,[IPC 147]
522,2013.INSC.754.txt,A.One Sunil (PW.1) lodged a complaint with th...,{'A.One Sunil (PW.1) lodged a complaint with t...,"[IPC 147, IPC 506]"
523,2016.INSC.13.txt,The case of the prosecution is that the appel...,{'On 2010 the appellant sexually violated the ...,"[IPC 376, IPC 506]"


In [21]:
train_df.at[524, 'explanation']

{'Accused no.l cut the left hand of Madhukar. ': 'IPC 302',
 'He also cut right foot of Madhukar. ': 'IPC 302',
 'The blow was given with 3 so much force that the blade of the axe stuck into the head of Madhukar and handle of the axe was broken. ': 'IPC 302',
 'His organs were severed by means of axes. ': 'IPC 302',
 'Reenabai (PW 3) tried to rescue her brother Madhukar, however, because of threats administered by the accused, she did not dare to rescue her brother Madhukar. ': 'IPC 506'}

In [22]:
path = 'task_1_statute_prediction.jsonl'
with open(path, 'r') as file:
    test_data = [json.loads(line) for line in file] 

In [23]:
test_df1 = pd.DataFrame(test_data)

In [24]:
test_df1

,id,fact,reasoning_traces,explanation
0,ST-PRED-0001,"Briefly stated, the case of the prosecution ag...",None,{'': []}
1,ST-PRED-0002,The case of the prosecution is that Keshari Na...,None,{'': []}
2,ST-PRED-0003,The occurrence is said to have taken place on ...,None,{'': []}
3,ST-PRED-0004,"The apple of discord, as revealed by the prose...",None,{'': []}
4,ST-PRED-0005,Bammiyampatti is a small village situated in t...,None,{'': []}
5,ST-PRED-0006,Put briefly the prosecution case is as follows...,None,{'': []}
6,ST-PRED-0007,Cherukuri Sambaiah was an affluent person havi...,None,{'': []}
7,ST-PRED-0008,The short facts of the case are that First Inf...,None,{'': []}
8,ST-PRED-0009,STATE OF KERALA [1997] INSC 829 (18 November 1...,None,{'': []}
9,ST-PRED-0010,VS [2000] INSC 298 (5 May 2000) Transfer Petit...,None,{'': []}


In [25]:
test_df = pd.DataFrame(test_data, columns=['id','fact','explanation'])

In [26]:
test_df

,id,fact,explanation
0,ST-PRED-0001,"Briefly stated, the case of the prosecution ag...",{'': []}
1,ST-PRED-0002,The case of the prosecution is that Keshari Na...,{'': []}
2,ST-PRED-0003,The occurrence is said to have taken place on ...,{'': []}
3,ST-PRED-0004,"The apple of discord, as revealed by the prose...",{'': []}
4,ST-PRED-0005,Bammiyampatti is a small village situated in t...,{'': []}
5,ST-PRED-0006,Put briefly the prosecution case is as follows...,{'': []}
6,ST-PRED-0007,Cherukuri Sambaiah was an affluent person havi...,{'': []}
7,ST-PRED-0008,The short facts of the case are that First Inf...,{'': []}
8,ST-PRED-0009,STATE OF KERALA [1997] INSC 829 (18 November 1...,{'': []}
9,ST-PRED-0010,VS [2000] INSC 298 (5 May 2000) Transfer Petit...,{'': []}


In [27]:
import numpy as np
import xgboost as xgb
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    f1_score, hamming_loss, average_precision_score, classification_report
)

In [28]:
from sklearn.metrics import precision_recall_curve

In [29]:
RNG = 42
EPS = 1e-7

In [30]:
class MultiLabelXGBSoftProb:
    def __init__(self, n_labels: int, use_softprob: bool = True,
                 auto_balance: bool = True, **xgb_params):
        self.n_labels = n_labels
        self.use_softprob = use_softprob
        self.auto_balance = auto_balance
        self.xgb_params = xgb_params
        self.models = []
        self.thresholds = None  # set by calibrate_thresholds(); falls back to 0.5 per label until then

    def fit(self, X, Y, eval_set=None):
        self.models = []
        for i in range(self.n_labels):
            y_i = Y[:, i]
            params = dict(self.xgb_params)

            if self.auto_balance and not self.use_softprob:
                n_pos = max(int(y_i.sum()), 1)
                n_neg = len(y_i) - n_pos
                params["scale_pos_weight"] = n_neg / n_pos

            if self.use_softprob:
                model = xgb.XGBClassifier(
                    objective=softprob_objective,
                    eval_metric="logloss",
                    base_score=0.0,
                    **params,
                )
            else:
                model = xgb.XGBClassifier(
                    objective="binary:logistic",
                    eval_metric="logloss",
                    **params,
                )

            model.fit(X, y_i)
            self.models.append(model)
        return self

    def predict_proba(self, X):
        proba = np.zeros((X.shape[0], self.n_labels))
        for i, model in enumerate(self.models):
            if self.use_softprob:
                raw = model.predict(X, output_margin=True)
                proba[:, i] = softprob(raw)
            else:
                proba[:, i] = model.predict_proba(X)[:, 1]
        return proba

    def calibrate_thresholds(self, X_val, Y_val):
        proba_val = self.predict_proba(X_val)
        thresholds = np.full(self.n_labels, 0.5)

        for i in range(self.n_labels):
            y_true = Y_val[:, i]
            if y_true.sum() == 0 or y_true.sum() == len(y_true):
                continue
            precision, recall, thresh = precision_recall_curve(y_true, proba_val[:, i])
            f1s = 2 * precision * recall / (precision + recall + 1e-9)
            if len(thresh) > 0:
                thresholds[i] = thresh[np.argmax(f1s[:-1])]

        self.thresholds = thresholds
        return thresholds

    def predict(self, X, threshold=None, ensure_at_least_one: bool = True):
        proba = self.predict_proba(X)

        if threshold is None:
            thresholds = self.thresholds if self.thresholds is not None else np.full(self.n_labels, 0.5)
        elif np.isscalar(threshold):
            thresholds = np.full(self.n_labels, threshold)
        else:
            thresholds = np.asarray(threshold)

        preds = (proba >= thresholds).astype(int)

        if ensure_at_least_one:
            empty_rows = preds.sum(axis=1) == 0
            if empty_rows.any():
                top_label_idx = proba[empty_rows].argmax(axis=1)
                preds[np.where(empty_rows)[0], top_label_idx] = 1

        return preds

    @staticmethod
    def _patch_xgb_base_score_for_shap(model):
        import json
        try:
            booster = model.get_booster()
            config = json.loads(booster.save_config())
            lmp = config.get("learner", {}).get("learner_model_param", {})
            raw = lmp.get("base_score")
            if isinstance(raw, str) and raw.strip().startswith("["):
                cleaned = raw.strip().strip("[]").split(",")[0]
                lmp["base_score"] = str(float(cleaned))
                booster.load_config(json.dumps(config))
        except Exception:
            pass
        return model

    def _feature_contributions(self, label_idx: int, doc_vec):
        model = self.models[label_idx]

        if getattr(self, "_shap_broken", False):
            return self._fallback_contributions(model, doc_vec)

        try:
            import shap
        except ImportError:
            self._shap_broken = True
            return self._fallback_contributions(model, doc_vec)

        if not hasattr(self, "_shap_explainers"):
            self._shap_explainers = {}

        if label_idx not in self._shap_explainers:
            self._patch_xgb_base_score_for_shap(model)
            try:
                self._shap_explainers[label_idx] = shap.TreeExplainer(model)
            except Exception as e:
                import warnings
                warnings.warn(
                    f"shap.TreeExplainer() failed ({type(e).__name__}: {e}). "
                    "Falling back to feature_importance*tfidf for all explanations in this run.",
                    RuntimeWarning,
                )
                self._shap_broken = True
                return self._fallback_contributions(model, doc_vec)

        explainer = self._shap_explainers[label_idx]
        try:
            shap_vals = explainer.shap_values(doc_vec)
        except Exception as e:
            import warnings
            warnings.warn(
                f"shap_values() failed ({type(e).__name__}: {e}); falling back for the rest of this run.",
                RuntimeWarning,
            )
            self._shap_broken = True
            return self._fallback_contributions(model, doc_vec)

        if isinstance(shap_vals, list):
            shap_vals = shap_vals[-1]
        contrib = np.asarray(shap_vals)[0]
        return contrib, "shap"

    def _fallback_contributions(self, model, doc_vec):
        dense = np.asarray(doc_vec.todense())[0]
        importances = model.feature_importances_
        contrib = dense * importances
        return contrib, "fallback"

    def explain(self, text: str, vectorizer, label_names,
                threshold=None, ensure_at_least_one: bool = True,
                max_sentences: int = 5, min_score_ratio: float = 0.4, top_n_terms: int = 5):
        import re

        sentences = [s.strip() for s in re.split(r"(?<=[.!?])\s+", text.strip()) if s.strip()]
        if not sentences:
            sentences = [text.strip()]

        doc_vec = vectorizer.transform([text])
        proba = self.predict_proba(doc_vec)[0]

        if threshold is None:
            thresholds = self.thresholds if self.thresholds is not None else np.full(self.n_labels, 0.5)
        elif np.isscalar(threshold):
            thresholds = np.full(self.n_labels, threshold)
        else:
            thresholds = np.asarray(threshold)

        predicted_idx = [i for i, p in enumerate(proba) if p >= thresholds[i]]
        if not predicted_idx and ensure_at_least_one:
            predicted_idx = [int(np.argmax(proba))]

        predicted = [label_names[i] for i in predicted_idx]

        analyzer = vectorizer.build_analyzer()
        vocab = vectorizer.vocabulary_

        explanations = {}
        for label in predicted:
            idx = label_names.index(label)
            contrib, _method = self._feature_contributions(idx, doc_vec)

            scored_sentences = []
            for sent in sentences:
                terms_in_sent = set(analyzer(sent))
                term_scores = []
                for term in terms_in_sent:
                    fidx = vocab.get(term)
                    if fidx is not None and contrib[fidx] > 0:
                        term_scores.append((term, float(contrib[fidx])))
                term_scores.sort(key=lambda t: -t[1])
                sent_score = sum(s for _, s in term_scores)
                scored_sentences.append({
                    "sentence": sent,
                    "score": sent_score,
                    "top_terms": term_scores[:top_n_terms],
                })

            scored_sentences.sort(key=lambda s: -s["score"])

            selected = []
            if scored_sentences and scored_sentences[0]["score"] > 0:
                top_score = scored_sentences[0]["score"]
                for s in scored_sentences:
                    if len(selected) >= max_sentences:
                        break
                    if s["score"] <= 0 or s["score"] < min_score_ratio * top_score:
                        break
                    selected.append(s)

            explanations[label] = selected

        return {
            "predicted_labels": predicted,
            "explanations": explanations,
            "probabilities": {label_names[i]: float(proba[i]) for i in range(self.n_labels)},
            "thresholds_used": {label_names[i]: float(thresholds[i]) for i in range(self.n_labels)},
        }

In [31]:
import ast

def parse_label_cell(cell, label_sep=","):
    """
    Parses a single Labels cell into a clean list of label strings.
    Handles two common formats:
 
      1. Plain delimited string:      "A,B"        -> ["A", "B"]
      2. Stringified Python list:     "['A', 'B']"  -> ["A", "B"]
    """
    s = str(cell).strip()
 
    try:
        val = ast.literal_eval(s)
        if isinstance(val, (list, tuple, set)):
            return [str(x).strip().strip("'\"") for x in val if str(x).strip()]
    except (ValueError, SyntaxError):
        pass
 
    # strip any stray brackets/quotes
    s = s.strip("[]")
    return [tok.strip().strip("'\"") for tok in s.split(label_sep) if tok.strip().strip("'\"")]



def load_dataset_from_dataframe(df, text_col="Text", label_col="Labels", label_sep=","):
   
    if text_col not in df.columns:
        raise ValueError(f"Expected text column '{text_col}', found {list(df.columns)}.")
    if label_col is not None and label_col not in df.columns:
        raise ValueError(
            f"Expected label column '{label_col}', found {list(df.columns)}. "
            f"Pass the actual column name via label_col=, or label_col=None if this "
            f"dataset has no ground-truth labels (inference only)."
        )
 
    subset_cols = [text_col] + ([label_col] if label_col is not None else [])
    df = df.dropna(subset=subset_cols).reset_index(drop=True)
 
    texts = df[text_col].astype(str).tolist()
    if label_col is not None:
        labels = [parse_label_cell(cell, label_sep=label_sep) for cell in df[label_col]]
    else:
        labels = [[] for _ in texts]
    return texts, labels

In [32]:
def labels_to_matrix(labels, label_names):
    Y = np.zeros((len(labels), len(label_names)), dtype=int)
    for i, doc_labels in enumerate(labels):
        for lab in doc_labels:
            Y[i, label_names.index(lab)] = 1
    return Y

In [33]:
def matrix_to_labels(Y, label_names):
    """Inverse of labels_to_matrix: multi-hot rows -> list of label-name lists."""
    return [[label_names[i] for i, v in enumerate(row) if v == 1] for row in Y]

In [34]:
def explain_test_set(model, texts, true_labels, vectorizer, label_names,
                      threshold: float = 0.5, top_n_sentences: int = 1,
                      top_n_terms: int = 5):
    """
    Runs model.explain() over every document in `texts` and returns a
    tidy DataFrame: one row per (document, predicted label, candidate
    explanation sentence).
    """
    import pandas as pd
 
    rows = []
    for text, true_labs in zip(texts, true_labels):
        result = model.explain(
            text, vectorizer, label_names,
            threshold=threshold, top_n_sentences=top_n_sentences, top_n_terms=top_n_terms,
        )
        predicted = result["predicted_labels"]
 
        if not predicted:
            rows.append({
                "text": text,
                "true_labels": ",".join(true_labs),
                "predicted_label": None,
                "explanation_sentence": None,
                "score": None,
                "top_terms": None,
            })
            continue
 
        for label in predicted:
            for exp in result["explanations"][label]:
                rows.append({
                    "text": text,
                    "true_labels": ",".join(true_labs),
                    "predicted_label": label,
                    "explanation_sentence": exp["sentence"],
                    "score": exp["score"],
                    "top_terms": exp["top_terms"],
                })
 
    return pd.DataFrame(rows)

In [35]:
import itertools
def flatten_column(df, column_name):
    if column_name in df.columns:
        # Flatten the lists in the specified column
        flattened = list(itertools.chain.from_iterable(df[column_name]))
        return flattened
    else:
        return f"Column '{column_name}' does not exist in the DataFrame."

flattened_values = flatten_column(train_df, 'statute')
#print(flattened_values)
unique_values = list(set(flattened_values))

print("Unique values:", unique_values)
print(len(unique_values))

Unique values: ['IPC 302', 'IPC 376', 'IPC 506', 'IPC 420', 'IPC 201', 'IPC 147', 'IPC 498A']
7


In [39]:
def _format_reasoning_trace(label_names, result):
    """
    Builds a free-text, step-by-step reasoning narrative from the
    structured output.
    """
    proba = result["probabilities"]
    thresholds = result["thresholds_used"]
    predicted = result["predicted_labels"]
    explanations = result["explanations"]
 
    lines = []
 
    prob_summary = ", ".join(f"{lab}={proba[lab]:.3f}" for lab in label_names)
    lines.append(f"Step 1: Computed label probabilities from the trained model: {prob_summary}.")
 
    crossed = [lab for lab in label_names if proba[lab] >= thresholds[lab]]
    if crossed:
        crossed_str = "; ".join(
            f"{lab} (p={proba[lab]:.3f} >= calibrated threshold {thresholds[lab]:.3f})" for lab in crossed
        )
        lines.append(f"Step 2: Label(s) crossing their calibrated decision threshold: {crossed_str}.")
    else:
        best = max(label_names, key=lambda lab: proba[lab])
        lines.append(
            f"Step 2: No label crossed its calibrated threshold. The highest-probability "
            f"label '{best}' (p={proba[best]:.3f}) was selected as the best available prediction "
            f"(fallback rule: always return at least one label)."
        )
 
    step = 3
    for label in predicted:
        exps = explanations.get(label, [])
        if not exps:
            continue
        top = exps[0]
        if top["top_terms"]:
            terms_str = ", ".join(f"'{t}' (contribution={s:.3f})" for t, s in top["top_terms"])
        else:
            terms_str = "no individually strong terms; contribution was spread across the sentence"
        lines.append(
            f"Step {step}: For label '{label}', the sentence most responsible for this prediction is: "
            f"\"{top['sentence']}\" (aggregate contribution score={top['score']:.3f}), driven mainly "
            f"by the term(s) {terms_str}."
        )
        step += 1
        if len(exps) > 1:
            others = "; ".join(f"\"{e['sentence']}\" (score={e['score']:.3f})" for e in exps[1:])
            lines.append(f"Step {step}: Additional supporting sentence(s) for '{label}': {others}.")
            step += 1
 
    return "\n".join(lines)

In [40]:
def generate_llm_reasoning(client, fact, predicted_labels, explanation_map,
                            model="gpt-4o-mini", provider="openai", max_tokens=400):
    """
    Uses an LLM API to write natural-language reasoning explaining WHY the
    already-predicted labels apply, grounded in the sentence-level
    evidence the trained model already identified (explanation_map).
    """
    if not predicted_labels:
        return "No label crossed the model's decision threshold for this text."

    evidence_lines = [f'- "{sent}" -> flagged for: {", ".join(labs)}'
                       for sent, labs in explanation_map.items()]
    evidence_block = "\n".join(evidence_lines) if evidence_lines else "(no sentence-level evidence extracted)"

    prompt = f"""You are explaining a text classification decision that has ALREADY been made by a trained model. Do not change, add, or second-guess the predicted labels below -- your only job is to explain, in natural language, how the identified sentences support them.

Text:
\"\"\"{fact}\"\"\"

Predicted label(s): {", ".join(predicted_labels)}

Sentence-level evidence identified by the model:
{evidence_block}

Write a short (3-5 sentence) natural-language explanation of how the evidence above supports the predicted label(s). Write in plain prose, not a numbered list or bullet points. Weave the reasoning naturally rather than mechanically restating the label list. Do not introduce any label not already listed above."""

    if provider == "openai":
        response = client.chat.completions.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return response.choices[0].message.content.strip()

    elif provider == "anthropic":
        response = client.messages.create(
            model=model,
            max_tokens=max_tokens,
            messages=[{"role": "user", "content": prompt}],
        )
        return "".join(block.text for block in response.content if block.type == "text").strip()

    else:
        raise ValueError(f"Unknown provider '{provider}'. Use 'openai' or 'anthropic'.")


In [41]:
def _write_records(records, path):
    import json
    if str(path).lower().endswith(".jsonl"):
        with open(path, "w", encoding="utf-8") as f:
            for rec in records:
                f.write(json.dumps(rec, ensure_ascii=False) + "\n")
    else:
        with open(path, "w", encoding="utf-8") as f:
            json.dump(records, f, indent=2, ensure_ascii=False)
 


In [42]:
def generate_explanation_json(model, texts, vectorizer, label_names, output_path,
                               ids=None, id_prefix="ST-PRED",
                               threshold=None, max_sentences: int = 5,
                               min_score_ratio: float = 0.4, top_n_terms: int = 5,
                               llm_client=None, llm_model: str = "gpt-4o-mini",
                               llm_provider: str = "openai"):
   
    import json
    import os
 
    if ids is None:
        ids = [f"{id_prefix}-{i + 1:04d}" for i in range(len(texts))]
    if len(ids) != len(texts):
        raise ValueError(f"ids length ({len(ids)}) must match texts length ({len(texts)})")
 
    output_path = os.path.abspath(output_path)
    out_dir = os.path.dirname(output_path)
    if out_dir:
        os.makedirs(out_dir, exist_ok=True)  # in case the target directory doesn't exist yet
 
    records = []
    try:
        for i, (doc_id, text) in enumerate(zip(ids, texts)):
            result = model.explain(
                text, vectorizer, label_names,
                threshold=threshold, max_sentences=max_sentences,
                min_score_ratio=min_score_ratio, top_n_terms=top_n_terms,
            )
 
            # sentence -> [labels]; a sentence can serve more than one label if
            # it was a top-contributing sentence for each of them
            explanation_map = {}
            for label, exps in result["explanations"].items():
                for exp in exps:
                    sent = exp["sentence"]
                    explanation_map.setdefault(sent, [])
                    if label not in explanation_map[sent]:
                        explanation_map[sent].append(label)
 
            if llm_client is not None:
                reasoning = generate_llm_reasoning(
                    llm_client, text, result["predicted_labels"], explanation_map,
                    model=llm_model, provider=llm_provider,
                )
            else:
                reasoning = _format_reasoning_trace(label_names, result)
 
            records.append({
                "id": doc_id,
                "fact": text,
                "reasoning_traces": reasoning,
                "explanation": explanation_map,
            })
 
            if (i + 1) % 10 == 0:
                print(f"  processed {i + 1}/{len(texts)} documents...")
 
    except Exception as e:
        
        partial_path = os.path.splitext(output_path)[0] + "_PARTIAL" + os.path.splitext(output_path)[1]
        _write_records(records, partial_path)
        print(f"FAILED after {len(records)}/{len(texts)} documents. "
              f"Partial results saved to: {partial_path}")
        raise
 
    _write_records(records, output_path)
 
   
    if not os.path.exists(output_path):
        raise RuntimeError(f"Write appeared to succeed but file not found at: {output_path}")
 
    size_kb = os.path.getsize(output_path) / 1024
    print(f"Wrote {len(records)} records to {output_path} ({size_kb:.1f} KB)")
    return records


In [43]:
def main(train_data_path: str = None, train_df=None,
         test_data_path: str = None, test_df=None,
         text_col: str = "fact", label_col: str = "statute",
         val_size: float = 0.15):
    # ---- load training data ----
    if train_df is not None:
        train_texts, train_labels = load_dataset_from_dataframe(train_df, text_col=text_col, label_col=label_col)
        print(f"Loaded {len(train_texts)} training documents from provided DataFrame")
    elif train_data_path:
        train_texts, train_labels = load_dataset_from_csv(train_data_path, text_col=text_col, label_col=label_col)
        print(f"Loaded {len(train_texts)} training documents from {train_data_path}")
    else:
        train_texts, train_labels = build_demo_dataset()
        print(f"No train_df/train_data_path given -- using {len(train_texts)}-document synthetic demo data")
 
    # ---- load test data ----
    # test_df/test_data_path may or may not have a ground-truth label
   
    if test_df is not None:
        test_has_labels = label_col in test_df.columns
        test_texts, test_labels = load_dataset_from_dataframe(
            test_df, text_col=text_col, label_col=(label_col if test_has_labels else None)
        )
        print(f"Loaded {len(test_texts)} test documents from provided DataFrame "
              f"(labels {'present -- will report metrics' if test_has_labels else 'NOT present -- inference only, no metrics'})")
    elif test_data_path:
        import pandas as pd
        _peek_cols = (pd.read_excel(test_data_path, nrows=0) if str(test_data_path).lower().endswith((".xlsx", ".xls"))
                      else pd.read_csv(test_data_path, nrows=0)).columns
        test_has_labels = label_col in _peek_cols
        test_texts, test_labels = load_dataset_from_csv(
            test_data_path, text_col=text_col, label_col=(label_col if test_has_labels else None)
        )
        print(f"Loaded {len(test_texts)} test documents from {test_data_path} "
              f"(labels {'present -- will report metrics' if test_has_labels else 'NOT present -- inference only, no metrics'})")
    else:
        raise ValueError(
            "No test data provided. Pass test_df=your_test_dataframe or "
            "test_data_path='path/to/test.csv'."
        )
 
    
    label_names = sorted({lab for doc in train_labels for lab in doc})
    print(f"Labels found in training data ({len(label_names)}): {label_names}")
 
    Y_train_full = labels_to_matrix(train_labels, label_names)
    # test_df's labels are converted with the SAFE variant: if test_df has
    # a label never seen in training, this warns and drops it instead of
    # crashing (labels_to_matrix would raise ValueError on an unseen label).
    # If test data has no labels at all (pure inference), Y_test is None.
    Y_test = labels_to_matrix_safe(test_labels, label_names, source_name="test data") if test_has_labels else None
 
    
    X_train_text, X_val_text, Y_train, Y_val = train_test_split(
        train_texts, Y_train_full, test_size=val_size, random_state=RNG
    )
    X_test_text = test_texts
    print(f"\nSplit: {len(X_train_text)} train / {len(X_val_text)} validation "
          f"(from train_df) / {len(X_test_text)} test (from test_df)")
 
    vectorizer = TfidfVectorizer(max_features=2000, ngram_range=(1, 2), stop_words="english")
    X_train = vectorizer.fit_transform(X_train_text)
    X_val = vectorizer.transform(X_val_text)
    X_test = vectorizer.transform(X_test_text)
 
    xgb_params = dict(
        n_estimators=200,
        max_depth=4,
        learning_rate=0.1,
        subsample=0.9,
        colsample_bytree=0.9,
        random_state=RNG,
        n_jobs=-1,
    )
 
    print("\n" + "=" * 60)
    print("Training baseline with standard sigmoid (binary:logistic)")
    print("=" * 60)
    model_sigmoid = MultiLabelXGBSoftProb(
        n_labels=len(label_names), use_softprob=False, **xgb_params
    )
    model_sigmoid.fit(X_train, Y_train)
 
    # calibrate per-label thresholds on validation data (carved out of
    # train_df, NOT test_df
    thresholds = model_sigmoid.calibrate_thresholds(X_val, Y_val)
    print("\nCalibrated per-label thresholds (vs flat 0.5 default):")
    for lab, t in zip(label_names, thresholds):
        print(f"  {lab}: {t:.3f}")
 
    proba_sig = model_sigmoid.predict_proba(X_test)
    pred_sig = model_sigmoid.predict(X_test)  # uses calibrated thresholds + at-least-one fallback automatically
 
    if test_has_labels:
        def report(name, y_true, y_pred, y_proba):
            print(f"\n--- {name} ---")
            print("Micro F1:      %.4f" % f1_score(y_true, y_pred, average="micro", zero_division=0))
            print("Macro F1:      %.4f" % f1_score(y_true, y_pred, average="macro", zero_division=0))
            print("Hamming Loss:  %.4f" % hamming_loss(y_true, y_pred))
            try:
                print("Avg Precision: %.4f" % average_precision_score(y_true, y_proba, average="macro"))
            except ValueError:
                pass  # can happen on small sets with a label that's all-0/all-1
            print(classification_report(y_true, y_pred, target_names=label_names, zero_division=0))
 
        report("Sigmoid (baseline)", Y_test, pred_sig, proba_sig)
    else:
        # no ground truth to score against, just summarize what got predicted
        n_labels_per_doc = pred_sig.sum(axis=1)
        print(f"\n--- Inference on test set (no ground-truth labels available -- no metrics) ---")
        print(f"Predicted label counts per document: min={n_labels_per_doc.min()}, "
              f"max={n_labels_per_doc.max()}, mean={n_labels_per_doc.mean():.2f}")
        for lab, count in zip(label_names, pred_sig.sum(axis=0)):
            print(f"  {lab}: predicted on {count}/{len(pred_sig)} documents")
 
    return model_sigmoid, vectorizer, label_names, X_test_text, Y_test  # Y_test is None if test data has no labels

In [44]:
# if __name__ == "__main__":
#     main(df=df)

model_sigmoid, vectorizer, label_names, X_test_text, Y_test = main(
    train_df=train_df,
    test_df=test_df,
    text_col="fact",
    label_col="statute",
)

Loaded 525 training documents from provided DataFrame
Loaded 57 test documents from provided DataFrame (labels NOT present -- inference only, no metrics)
Labels found in training data (7): ['IPC 147', 'IPC 201', 'IPC 302', 'IPC 376', 'IPC 420', 'IPC 498A', 'IPC 506']

Split: 446 train / 79 validation (from train_df) / 57 test (from test_df)

Training baseline with standard sigmoid (binary:logistic)

Calibrated per-label thresholds (vs flat 0.5 default):
  IPC 147: 0.042
  IPC 201: 0.179
  IPC 302: 0.597
  IPC 376: 0.707
  IPC 420: 0.679
  IPC 498A: 0.430
  IPC 506: 0.083

--- Inference on test set (no ground-truth labels available -- no metrics) ---
Predicted label counts per document: min=1, max=4, mean=1.46
  IPC 147: predicted on 27/57 documents
  IPC 201: predicted on 6/57 documents
  IPC 302: predicted on 19/57 documents
  IPC 376: predicted on 6/57 documents
  IPC 420: predicted on 8/57 documents
  IPC 498A: predicted on 9/57 documents
  IPC 506: predicted on 8/57 documents


In [46]:
# Check what parameter is configured
print(model_sigmoid.get_params().get("multi_strategy"))

AttributeError: 'MultiLabelXGBSoftProb' object has no attribute 'get_params'

In [51]:
# pip install openai

In [52]:
from openai import OpenAI

In [53]:
import os

In [54]:
os.environ["OPENAI_API_KEY"] = "sk-.............."   # insert your own API Key instead of "sk-.............." 
client = OpenAI(api_key=os.environ.get("OPENAI_API_KEY"))

In [76]:
records = generate_explanation_json(
    model_sigmoid,
    X_test_text,
    vectorizer,
    label_names,
    output_path="predictions_with_explanations_final.jsonl",   # <-- .jsonl extension triggers line-delimited output
    ids=test_df["id"].astype(str).tolist(),
    threshold=None,
    max_sentences=5,        # ceiling, not a fixed target
    min_score_ratio=0.8,    # tune this if results feel too loose/strict
    llm_client=client,
    llm_model="gpt-4o-mini",
    llm_provider="openai",
)

  processed 10/57 documents...
  processed 20/57 documents...
  processed 30/57 documents...
  processed 40/57 documents...
  processed 50/57 documents...
Wrote 57 records to /home/vnit/Downloads/FIRE/predictions_with_explanations_final.jsonl (554.6 KB)


In [77]:
path = 'predictions_with_explanations_final.jsonl'
with open(path, 'r') as file:
    result_data = [json.loads(line) for line in file] 

In [78]:
result_df = pd.DataFrame(result_data)

In [79]:
# result_df

In [97]:
result_df.at[52,'explanation']

{'It is the case of the appellant that she married to respondent no.1 on July 8, After the marriage, she remained with her husband for few days at Jabalpur and during that period, her husband and in-laws harassed her as her father had not given sufficient amount of dowry.': ['IPC 498A'],
 'On September 28, 1996, challan was filed against the respondents for offences punishable under Sections 498A, 506, 406 read with Section 34 of Indian Indian Penal Code, 1860, 1860 (Indian Penal Code, 1860) and also under Sections 3 and 4 of [Dowry Prohibition Act, 1961](http://www.liiofindia.org/in/legis/cen/num_act/dpa1961225/).': ['IPC 498A',
  'IPC 506'],
 'The appellant informed her father that her husband and in- laws were demanding dowry from her and her husband assaulted her and her children had been taken away and they were not allowed to see the mother (appellant).': ['IPC 498A'],
 'They taunted the appellant saying that had the respondent no.1 married to any other lady, they would have rece

In [98]:
result_df.at[52,'reasoning_traces']

"The evidence provided strongly supports the application of IPC 498A, which addresses cruelty by a husband or his relatives towards a wife, as it details the appellant's experiences of harassment and abuse by her husband and in-laws connected to dowry demands. The statement about the in-laws taunting the appellant regarding dowry expectations further illustrates the emotional and psychological cruelty she faced. Additionally, the appellant’s allegation that her husband assaulted her and deprived her of her children underscores the severity of her suffering, which is central to the charges under IPC 498A. The mention of a formal complaint filed against the respondents highlights the legal recognition of such abuses, while the filing of a challan including IPC 506 indicates the threats and intimidation involved in her situation, further supporting the charges of cruelty and coercion."